# 🎲 Embracing Chaos: The Bayesian PINNsFormer
Now that we have built a robust deterministic engine, we must address its fundamental flaw: **Overconfidence**. 

In highly non-linear, stiff systems like the Hodgkin-Huxley equations, the exact timing of an action potential is deeply sensitive. A deterministic model will always predict a spike at exactly one timestamp. If the model is slightly unsure due to sparse data or complex physics, it doesn't tell us—it just outputs a single, highly confident (and potentially wrong) line.

To solve this, we are upgrading to a **Bayesian PINNsFormer**. We are shifting our goal from predicting a single "best guess" trajectory to predicting a *distribution* of possible trajectories, allowing us to quantify **Epistemic Uncertainty** (the model's uncertainty in its own knowledge).

## 1. The Mathematical Shift: Monte Carlo Dropout
In a deterministic network, the weights $\theta$ are fixed numbers, mapping inputs strictly to outputs:
$$ U_{pred} = f(T_{seq}, \theta) $$

In a True Bayesian Neural Network (BNN), every weight is replaced by a probability distribution (e.g., $P(\theta \mid \mathcal{D})$). However, calculating a distribution for every single parameter in a Transformer is computationally impossible.

### The MC-Dropout Shortcut
To approximate a Bayesian network without the explosive computational cost, we use a mathematically rigorous shortcut discovered by Yarin Gal and Zoubin Ghahramani: **Monte Carlo Dropout (MC-Dropout)**.

Normally, dropout randomly zeroes out neurons during training to prevent overfitting and is turned *off* during inference. **MC-Dropout keeps dropout turned ON during inference.** 

Applying dropout is mathematically equivalent to multiplying the weight matrix by a Bernoulli random variable. Every time we pass data through the network, we are sampling a slightly different "sub-network." This perfectly approximates sampling from the posterior distribution of the weights:
$$ y \sim \text{Transformer}(x, \text{Dropout}(\theta)) $$

## 2. Modifying the Architecture
To make our PINNsFormer Bayesian, we strategically inject Dropout layers into the architecture we built previously. We place them in two critical locations:

1.  **After the Self-Attention Mechanism:** This simulates uncertainty in how different time steps relate to each other globally.
2.  **Inside the Wavelet Feed-Forward Network:** This simulates uncertainty in the exact amplitude and strictness of the localized voltage spikes.

When an input sequence passes through this network, the Mexican Hat wavelets still capture the stiff, sharp dynamics, but the dropout layers slightly shift the timing and amplitude of those activations.

## 2. Modifying the Architecture
To make our PINNsFormer Bayesian, we strategically inject Dropout layers into the architecture we built previously. We place them in two critical locations:

1.  **After the Self-Attention Mechanism:** This simulates uncertainty in how different time steps relate to each other globally.
2.  **Inside the Wavelet Feed-Forward Network:** This simulates uncertainty in the exact amplitude and strictness of the localized voltage spikes.

When an input sequence passes through this network, the Mexican Hat wavelets still capture the stiff, sharp dynamics, but the dropout layers slightly shift the timing and amplitude of those activations.

## 3. Inference: Extracting Mean and Variance
Because the network is now non-deterministic, running it once is meaningless (you just get one random sample). Instead, we perform **Monte Carlo Sampling**. We run the exact same input sequence $T_{seq}$ through the network $M$ times (usually $M = 50$ to $100$ passes). Let $\hat{U}_m(t)$ be the network's prediction at pass $m$.

### A. The Predicted Trajectory (Empirical Mean)
The final, most likely trajectory is the average of all $M$ forward passes:
$$ \mu(t) = \frac{1}{M} \sum_{m=1}^M \hat{U}_m(t) $$

### B. The Epistemic Uncertainty (Variance)
To calculate the confidence bands (how much the predictions fluctuate), we calculate the variance across the $M$ passes:
$$ \sigma^2(t) = \frac{1}{M} \sum_{m=1}^M \left( \hat{U}_m(t) - \mu(t) \right)^2 $$

*When plotted, the empirical mean acts as the solid central line, surrounded by a shaded ribbon representing $\mu(t) \pm 2\sigma(t)$ (the 95% confidence interval). This ribbon remains narrow during resting potentials and bulges out exactly where the spikes occur, visualizing the model's uncertainty in spike timing.*

## 4. The Bayesian Loss Function Synergy
The most beautiful aspect of MC-Dropout is that our physics-informed loss function does not actually need to change:
$$ \mathcal{L}_{total} = \mathcal{L}_{data} + \lambda \mathcal{L}_{phys} $$

In variational Bayesian inference, the loss function usually requires a highly complex "KL-Divergence" term to keep the weight distributions mathematically sensible. However, it has been proven that standard $L_2$ regularization (Weight Decay), combined with the random masking of Dropout, is exactly mathematically equivalent to this KL-Divergence term.

We simply train the network normally with Dropout enabled. The network naturally learns to minimize the finite-difference physics residuals across a massive distribution of sub-networks, creating a highly robust, uncertainty-aware physics solver.

## 4. The Bayesian Loss Function Synergy
The most beautiful aspect of MC-Dropout is that our physics-informed loss function does not actually need to change:
$$ \mathcal{L}_{total} = \mathcal{L}_{data} + \lambda \mathcal{L}_{phys} $$

In variational Bayesian inference, the loss function usually requires a highly complex "KL-Divergence" term to keep the weight distributions mathematically sensible. However, it has been proven that standard $L_2$ regularization (Weight Decay), combined with the random masking of Dropout, is exactly mathematically equivalent to this KL-Divergence term.

We simply train the network normally with Dropout enabled. The network naturally learns to minimize the finite-difference physics residuals across a massive distribution of sub-networks, creating a highly robust, uncertainty-aware physics solver.

In [ ]:
# %% Cell 1: Environment Setup
using Lux, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote

# Assuming df_ordered, t_train, z_train, and scale_factors are loaded
# Shape formatting for Transformer: (feature_dim, seq_len, batch_size)
t_seq = reshape(t_train, 1, length(t_train), 1) # (1, N, 1)
z_seq = reshape(z_train, 4, length(t_train), 1) # (4, N, 1)

dt = Float32(t_train[2] - t_train[1])

rng = Random.default_rng()
Random.seed!(rng, 42)

In [ ]:
# %% Cell 2: Architecture with MC-Dropout
# 1. Mexican Hat Wavelet Activation
mexican_hat(x) = (1.0f0 .- x.^2) .* exp.(-0.5f0 .* x.^2)

# 2. Custom Self-Attention Wrapper
struct SelfAttention{M} <: Lux.AbstractExplicitLayer
    mha::M
end
SelfAttention(d_model, n_heads) = SelfAttention(Lux.MultiHeadAttention(d_model, n_heads))

function ((layer::SelfAttention)(x, ps, st::NamedTuple))
    out, st_mha = layer.mha((x, x, x), ps, st)
    return out[1], st_mha 
end

# 3. Hyperparameters
d_model = 32     
n_heads = 4      
d_ff = 64        
dropout_rate = 0.1f0 # 10% dropout rate for Bayesian uncertainty

# 4. Build the Bayesian PINNsFormer
pinnsformer = Lux.Chain(
    Lux.Dense(1 => d_model),
    
    # Block A: Self-Attention with Dropout
    Lux.SkipConnection(
        Lux.Chain(
            Lux.LayerNorm((d_model,)),
            SelfAttention(d_model, n_heads),
            Lux.Dropout(dropout_rate) # Epistemic uncertainty in temporal relations
        ),
        +
    ),
    
    # Block B: Wavelet FFN with Dropout
    Lux.SkipConnection(
        Lux.Chain(
            Lux.LayerNorm((d_model,)),
            Lux.Dense(d_model => d_ff),
            Lux.WrappedFunction(mexican_hat),
            Lux.Dropout(dropout_rate), # Epistemic uncertainty in spike amplitude/timing
            Lux.Dense(d_ff => d_model),
            Lux.Dropout(dropout_rate)
        ),
        +
    ),
    
    Lux.Dense(d_model => 4)
)

ps, st = Lux.setup(rng, pinnsformer)
p_nn = ComponentArray(ps)

In [ ]:
# %% Cell 3: Loss Functions (Identical to Deterministic)
# (Assumes your hh_equations(u) function is defined)

function physics_loss_seq(u_pred, dt_step, s_factors)
    # Central difference approximation
    u_next = u_pred[:, 3:end, :]
    u_prev = u_pred[:, 1:end-2, :]
    du_dt_pred = (u_next .- u_prev) ./ (2.0f0 * dt_step)
    
    # True HH derivatives
    u_interior = u_pred[:, 2:end-1, :]
    u_interior_2d = reshape(u_interior, 4, :)
    true_derivs_2d = hh_equations(u_interior_2d)
    true_derivs_seq = reshape(true_derivs_2d, 4, size(u_interior, 2), 1)
    
    # Residuals
    residuals = (du_dt_pred .- true_derivs_seq) ./ s_factors
    return mean(abs2, residuals)
end

function total_loss(p, _)
    # Forward pass: st updates the dropout random state on each pass
    u_pred, _ = pinnsformer(t_seq, p, st)
    
    loss_data = mean(abs2, u_pred .- z_seq)
    loss_phys = physics_loss_seq(u_pred, dt, scale_factors)
    
    λ = 1.5f0
    return loss_data + λ * loss_phys
end

In [ ]:
# %% Cell 4: Training Optimization
loss_history = Float32[]

callback_fn = function (p, l)
    push!(loss_history, l)
    if length(loss_history) % 50 == 0
        println("Epoch $(length(loss_history)) | Total Loss: $(round(l, digits=5))")
    end
    return false 
end

opt_func = OptimizationFunction(total_loss, Optimization.AutoZygote())
opt_prob = OptimizationProblem(opt_func, p_nn)

println("Starting Bayesian PINNsFormer Training...")
result = solve(opt_prob, Adam(0.0005), maxiters = 3000, callback = callback_fn)
println("Training Complete!")

p_opt = result.u

In [ ]:
# %% Cell 5: Monte Carlo Dropout Inference
num_samples = 50
predictions = zeros(Float32, 4, length(t_train), num_samples)

# We use the training state `st` (which has dropout ACTIVE)
# Do NOT use Lux.testmode(st) here!
st_infer = st 

println("Running $num_samples Monte Carlo passes for uncertainty quantification...")

for i in 1:num_samples
    # The updated st_infer guarantees a new dropout mask for this pass
    out_seq, st_infer = pinnsformer(t_seq, p_opt, st_infer)
    
    # Reshape (4, N, 1) -> (4, N) and store
    predictions[:, :, i] = reshape(out_seq, 4, length(t_train))
end

# Calculate the Empirical Mean (The predicted trajectory)
mean_trajectory = dropdims(mean(predictions, dims=3), dims=3)

# Calculate the Epistemic Uncertainty (Standard Deviation)
std_trajectory = dropdims(std(predictions, dims=3), dims=3)

# %% Plotting the Results with Confidence Bands
using Plots

v_mean = mean_trajectory[1, :]
v_std = std_trajectory[1, :]

# We plot the mean, and use ribbon to plot ± 2 standard deviations (approx 95% confidence)
plot(t_train, v_mean, 
    ribbon = 2 .* v_std, 
    fillalpha = 0.3, # Transparency for the confidence band
    title = "Bayesian PINNsFormer: Voltage Prediction",
    xlabel = "Time (ms)",
    ylabel = "Voltage (mV)",
    color = :cyan,
    linewidth = 2,
    label = "Predicted Mean (V)",
    grid = true
)